# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).**

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`.

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]

    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()

    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))

    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")

    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Małe zbrodnie małżeńskie
2. Jozef Van Wissem: Gabinet doktora Caligari w Gwarku
3. To wiem na pewno
4. Piotr Bałtroczyk w Kinie Kijów
5. #Osiecka
6. Kabaret hrAbi: Być facetem
7. Uśmiechnij się Mamo
8. Gdy Piwnica się rodziła
9. Mały Książę (Teatr Ludowy)
10. Tajemnice Buenos Aires

Czas wykonania: 5.52s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]

    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        results = list(executor.map(download_site, sites))

    all_titles = [title for sublist in results for title in sublist]

    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")

    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Małe zbrodnie małżeńskie
2. Jozef Van Wissem: Gabinet doktora Caligari w Gwarku
3. To wiem na pewno
4. Piotr Bałtroczyk w Kinie Kijów
5. #Osiecka
6. Kabaret hrAbi: Być facetem
7. Uśmiechnij się Mamo
8. Gdy Piwnica się rodziła
9. Tajemnice Buenos Aires
10. Mały Książę (Teatr Ludowy)

Czas wykonania (wątki): 2.17s


---
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [3]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))

print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


---
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [4]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()

    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)

    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 2 procesach (rdzeniach)...
Zakończono w czasie 3.37s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [7]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def get_cat_fact():
    """Pobiera jeden fakt o kotach."""
    response = requests.get(CAT_API_URL)
    response.raise_for_status()
    return response.json().get('fact')

def sequential_fetch():
    print("=== SEKWENCYJNE pobieranie 20 faktów ===")
    start = time.time()
    facts = []
    for _ in range(20):
        facts.append(get_cat_fact())
    end = time.time()
    print(f"Pobrano {len(facts)} faktów w {end - start:.2f} sekund.")
    print("Pierwsze 3 fakty:")
    for i, fact in enumerate(facts[:3], 1):
        print(f"{i}. {fact}")
    return facts

def threaded_fetch():
    print("\n=== WIELOWĄTKOWE pobieranie 20 faktów ===")
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        facts = list(executor.map(lambda _: get_cat_fact(), range(20)))

    end = time.time()
    print(f"Pobrano {len(facts)} faktów w {end - start:.2f} sekund.")
    print("Pierwsze 3 fakty:")
    for i, fact in enumerate(facts[:3], 1):
        print(f"{i}. {fact}")
    return facts

if __name__ == "__main__":
    sequential_fetch()
    threaded_fetch()

=== SEKWENCYJNE pobieranie 20 faktów ===
Pobrano 20 faktów w 1.92 sekund.
Pierwsze 3 fakty:
1. Cats have the largest eyes of any mammal.
2. Cats are now Britain's favourite pet: there are 7.7 million cats as opposed to 6.6 million dogs.
3. Cats, just like people, are subject to asthma. Dust, smoke, and other forms of air pullution in your cat's environment can be troublesome sources of irritation.

=== WIELOWĄTKOWE pobieranie 20 faktów ===
Pobrano 20 faktów w 0.32 sekund.
Pierwsze 3 fakty:
1. Cats eat grass to aid their digestion and to help them get rid of any fur in their stomachs.
2. Cats' eyes shine in the dark because of the tapetum, a reflective layer in the eye, which acts like a mirror.
3. The cat's footpads absorb the shocks of the landing when the cat jumps.


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [8]:
import queue
import threading
import time

def producer(q, num_items=50):
    """Producent generuje liczby naturalne."""
    for i in range(1, num_items + 1):
        q.put(i)
        time.sleep(0.01)  # symulacja pracy

    # Sygnały zakończenia dla konsumentów
    q.put(None)
    q.put(None)
    print("Producent zakończył generowanie liczb.")

def consumer_even(q, results):
    """Konsument parzystych liczb."""
    while True:
        item = q.get()
        if item is None:
            q.put(None)  # przekaż sygnał dalej
            break
        if item % 2 == 0:
            results.append(item)
            print(f"Konsument parzysty: {item}")
        q.task_done()

def consumer_odd(q, results):
    """Konsument nieparzystych liczb."""
    while True:
        item = q.get()
        if item is None:
            q.put(None)
            break
        if item % 2 != 0:
            results.append(item)
            print(f"Konsument nieparzysty: {item}")
        q.task_done()

if __name__ == "__main__":
    q = queue.Queue(maxsize=20)
    even_results = []
    odd_results = []

    # Tworzenie wątków
    prod_thread = threading.Thread(target=producer, args=(q, 50))
    even_thread = threading.Thread(target=consumer_even, args=(q, even_results))
    odd_thread = threading.Thread(target=consumer_odd, args=(q, odd_results))

    start = time.time()

    prod_thread.start()
    even_thread.start()
    odd_thread.start()

    prod_thread.join()
    even_thread.join()
    odd_thread.join()

    end = time.time()

    print(f"\n=== PODSUMOWANIE ===")
    print(f"Liczb parzystych: {len(even_results)}")
    print(f"Liczb nieparzystych: {len(odd_results)}")
    print(f"Czas wykonania: {end - start:.2f} sekund")

Konsument parzysty: 2
Konsument nieparzysty: 3
Konsument parzysty: 4
Konsument nieparzysty: 5
Konsument parzysty: 6
Konsument nieparzysty: 7
Konsument parzysty: 8
Konsument nieparzysty: 9
Konsument parzysty: 10
Konsument nieparzysty: 11
Konsument parzysty: 12
Konsument nieparzysty: 13
Konsument parzysty: 14
Konsument nieparzysty: 15
Konsument parzysty: 16
Konsument nieparzysty: 17
Konsument parzysty: 18
Konsument nieparzysty: 19
Konsument parzysty: 20
Konsument nieparzysty: 21
Konsument parzysty: 22
Konsument nieparzysty: 23
Konsument parzysty: 24
Konsument nieparzysty: 25
Konsument parzysty: 26
Konsument nieparzysty: 27
Konsument parzysty: 28
Konsument nieparzysty: 29
Konsument parzysty: 30
Konsument nieparzysty: 31
Konsument parzysty: 32
Konsument nieparzysty: 33
Konsument parzysty: 34
Konsument nieparzysty: 35
Konsument parzysty: 36
Konsument nieparzysty: 37
Konsument parzysty: 38
Konsument nieparzysty: 39
Konsument parzysty: 40
Konsument nieparzysty: 41
Konsument parzysty: 42
Konsu

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [9]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

def run_power_sum_demo():
    print("=== Wieloprocesowe obliczanie sumy potęg ===")
    start_total = time.time()

    numbers = list(range(1, 10001))  # 1 do 10 000
    num_processes = multiprocessing.cpu_count()
    print(f"Używam {num_processes} procesów.")

    with multiprocessing.Pool(processes=num_processes) as pool:
        start = time.time()
        results = pool.map(calculate_power_sum, numbers)
        end = time.time()

    print(f"Obliczenia zakończone w {end - start:.2f} sekund.")
    print(f"Przetworzono {len(results)} liczb.")
    print(f"Czas całkowity programu: {time.time() - start_total:.2f} sekund.")

if __name__ == "__main__":
    run_power_sum_demo()

=== Wieloprocesowe obliczanie sumy potęg ===
Używam 2 procesów.
Obliczenia zakończone w 0.69 sekund.
Przetworzono 10000 liczb.
Czas całkowity programu: 0.72 sekund.
